In [7]:
import numpy as np
from scipy.optimize import linprog
import time
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict
import heapq
import random

In [8]:
@dataclass
class MKPInstance:
    n: int                        # số vật phẩm
    m: int                        # số ràng buộc (chiều)
    profits: np.ndarray           # lợi nhuận pj,  shape (n,)
    weights: np.ndarray           # trọng số wi,j, shape (m, n)
    capacities: np.ndarray        # sức chứa ci,   shape (m,)

    def __post_init__(self):
        self.profits   = np.asarray(self.profits,    dtype=float)
        self.weights   = np.asarray(self.weights,    dtype=float)
        self.capacities= np.asarray(self.capacities, dtype=float)

In [9]:
@dataclass
class State:
    weight: float
    profit: float
    res: np.ndarray
    items: frozenset = field(default_factory=frozenset)
    level: int = 0
    ub: float = 0.0

    def dominates(self, other: "State") -> bool:
        return self.profit >= other.profit and self.weight <= other.weight

    def is_feasible(self, inst: MKPInstance) -> bool:
        return bool(np.all(self.res >= 0))

In [10]:
#  GIAI ĐOẠN 1: NỚI LỎNG SURROGATE

def surrogate_relaxation(inst: MKPInstance) -> Tuple[np.ndarray, np.ndarray, float]:
    print("    Giai đoạn 1: Nới lỏng Surrogate ")

    n, m = inst.n, inst.m
    p = inst.profits
    W = inst.weights   # (m, n)
    c = inst.capacities

    res_lp = linprog(
        c       = -p,
        A_ub    = W,
        b_ub    = c,
        bounds  = [(0, 1)] * n,
        method  = "highs"
    )

    if res_lp.status not in (0, 1):
        print("  [Warning] LP không hội tụ, dùng trọng số đều u_i = 1/m")
        u = np.ones(m) / m
    else:
        try:
            u = -res_lp.ineqlin.marginals          # >= 0 vì min -p
        except AttributeError:
            u = np.ones(m) / m

        u = np.maximum(u, 0)
        epsilon = 1e-4
        u = u + epsilon
        u = u / u.sum()
        if u.sum() < 1e-9:
            u = np.ones(m) / m
        else:
            u = u / u.sum()

    w_surrogate = u @ W
    C_surrogate = float(u @ c)

    print(f"    Dual multipliers u = {np.round(u, 4)}")
    print(f"    Sức chứa surrogate C = {C_surrogate:.4f}")
    return u, w_surrogate, C_surrogate

In [11]:
def get_efficiency_order(inst, w_surr):
    eff = inst.profits / (w_surr + 1e-9)
    return np.argsort(-eff)

In [12]:
#  TÍNH CẬN TRÊN (Upper Bound) – Fractional Knapsack

def upper_bound_fast(state, inst, w_surr, C_surr_global, sorted_indices):
    rem_w = C_surr_global - state.weight
    if rem_w <= 1e-9:
        return state.profit

    ub_profit = state.profit
    for j in sorted_indices:
        if j < state.level: continue
        if j in state.items: continue

        if w_surr[j] <= rem_w + 1e-9:
            ub_profit += inst.profits[j]
            rem_w -= w_surr[j]
        else:
            ub_profit += (rem_w / w_surr[j]) * inst.profits[j]
            break
    return ub_profit

In [13]:
#  GIAI ĐOẠN 2: LẬP TRÌNH ĐỘNG (DP)

def dominance_check(states: List[State]) -> List[State]:
    if not states:
        return states

    states.sort(key=lambda s: (s.weight, -s.profit))

    kept = []
    best_profit = -np.inf
    for s in states:
        if s.profit > best_profit:
            kept.append(s)
            best_profit = s.profit

    return kept


In [14]:
def dynamic_programming_optimized(inst, w_surr, C_surr, lb, stop_item, max_beam_size=5000):
    print(f"    Giai đoạn 2: DP Tối ưu (Beam Size: {max_beam_size})")

    sorted_idx = get_efficiency_order(inst, w_surr)
    init_state = State(weight=0.0, profit=0.0, res=inst.capacities.copy(), items=frozenset(), level=0)
    L = [init_state]
    best_profit = lb
    best_items = frozenset()
    print_step = max(1, stop_item // 5)

    for j in range(stop_item):
        pj, wj = inst.profits[j], w_surr[j]
        wj_orig = inst.weights[:, j]
        L_new = []

        for s in L:
            s_no = State(s.weight, s.profit, s.res, s.items, j + 1)
            L_new.append(s_no)

            new_res = s.res - wj_orig
            if np.all(new_res >= 0):
                new_w = s.weight + wj
                if new_w <= C_surr + 1e-9:
                    new_p = s.profit + pj
                    new_state = State(new_w, new_p, new_res, s.items | {j}, j + 1)
                    if new_p > best_profit:
                        best_profit, best_items = new_p, new_state.items
                    L_new.append(new_state)

        L_temp = dominance_check(L_new)
        L_next = []
        for s in L_temp:
            ub = upper_bound_fast(s, inst, w_surr, C_surr, sorted_idx)
            if ub > best_profit + 1e-7:
                s.ub = ub
                L_next.append(s)

        if len(L_next) > max_beam_size:
            L_next.sort(key=lambda x: x.ub, reverse=True)
            L_next = L_next[:max_beam_size]

        L = L_next
        if (j + 1) % print_step == 0 or (j + 1) == stop_item:
            print(f"    Tiến độ: {j+1}/{stop_item} | Số trạng thái: {len(L)} | Best: {best_profit}")

    return L, best_profit, best_items

In [15]:
#  GIAI ĐOẠN 3: BRANCH AND BOUND

import heapq

def branch_and_bound_optimized(inst, candidates, w_surr, lb, initial_items,
                               C_surr_global, sorted_idx, time_limit=60.0):
    print(f"    Giai đoạn 3: B&B Tối ưu (Best-First Search)")

    start_time = time.time()
    best_profit = lb
    best_items = initial_items
    nodes_explored = 0

    count = 0
    pq = []
    for s in candidates:
        ub = upper_bound_fast(s, inst, w_surr, C_surr_global, sorted_idx)
        if ub > best_profit + 1e-7:
            heapq.heappush(pq, (-ub, s.profit, s.weight, count, s.res, s.level, s.items))
            count += 1

    while pq:
        if time.time() - start_time > time_limit:
            print("    [Cảnh báo] Đã đạt giới hạn thời gian B&B.")
            break

        neg_ub, curr_profit, curr_w, _, curr_res, level, curr_items = heapq.heappop(pq)
        ub = -neg_ub
        nodes_explored += 1
        if ub <= best_profit + 1e-7:
            continue

        if level >= inst.n:
            if curr_profit > best_profit:
                best_profit = curr_profit
                best_items = curr_items
            continue

        j = level

        # --- NHÁNH 1: CHỌN vật phẩm j ---
        pj = inst.profits[j]
        wj_surr = w_surr[j]
        wj_orig = inst.weights[:, j]

        new_res = curr_res - wj_orig
        if np.all(new_res >= 0) and (curr_w + wj_surr <= C_surr_global + 1e-9):
            new_profit = curr_profit + pj
            new_items = curr_items | {j}

            if new_profit > best_profit:
                best_profit = new_profit
                best_items = new_items

            temp_state = State(curr_w + wj_surr, new_profit, new_res, new_items, j + 1)
            new_ub = upper_bound_fast(temp_state, inst, w_surr, C_surr_global, sorted_idx)

            if new_ub > best_profit + 1e-7:
                heapq.heappush(pq, (-new_ub, new_profit, curr_w + wj_surr, count, new_res, j + 1, new_items))
                count += 1

        # --- NHÁNH 2: KHÔNG CHỌN vật phẩm j ---
        temp_state_no = State(curr_w, curr_profit, curr_res, curr_items, j + 1)
        no_ub = upper_bound_fast(temp_state_no, inst, w_surr, C_surr_global, sorted_idx)

        if no_ub > best_profit + 1e-7:
            heapq.heappush(pq, (-no_ub, curr_profit, curr_w, count, curr_res, j + 1, curr_items))
            count += 1

    print(f"    B&B Hoàn tất: Lợi nhuận = {best_profit:.2f}, Nodes = {nodes_explored}")
    return best_profit, best_items

In [35]:
#  HÀM CHÍNH: THUẬT TOÁN HDP

def hdp_solve(inst: MKPInstance,
              time_limit: float = 600.0,
              bb_tail_items: int = 20) -> Dict:

    t0 = time.time()

    # Giai đoạn 1: Surrogate Relaxation
    u, w_surr, C_surr = surrogate_relaxation(inst)

    sorted_idx = get_efficiency_order(inst, w_surr)

    lb = 0.0

    # Giai đoạn 2: DP
    stop_item = max(0, inst.n - max(0, bb_tail_items))
    L_candidates, lb, lb_items = dynamic_programming_optimized(
        inst, w_surr, C_surr, lb, stop_item=stop_item, max_beam_size=10000
    )

    # Giai đoạn 3: B&B
    remaining_time = max(0.0, time_limit - (time.time() - t0))
    best_profit, best_items = branch_and_bound_optimized(
        inst,
        candidates=L_candidates,
        w_surr=w_surr,
        lb=lb,
        initial_items=lb_items,
        C_surr_global=C_surr,
        sorted_idx=sorted_idx,
        time_limit=remaining_time
    )


    x = np.zeros(inst.n, dtype=int)
    for j in best_items:
        x[j] = 1

    feasible = bool(np.all(inst.weights @ x <= inst.capacities))
    real_profit = float(inst.profits @ x)

    elapsed = time.time() - t0
    print()
    print(f"  KẾT QUẢ CUỐI CÙNG")
    print(f"  Lợi nhuận   : {real_profit:.2f}")
    print(f"  Khả thi     : {'Có' if feasible else 'Không'}")
    print(f"  Thời gian   : {elapsed:.3f}s")


    return {
        "profit"         : real_profit,
        "items"          : sorted(best_items),
        "solution_vector": x.tolist(),
        "feasible"       : feasible,
        "time_sec"       : elapsed,
    }

In [36]:
import numpy as np
def solve_from_uploaded_file(file_path="mknapcb1.txt", instance_to_solve=1):
    try:
        with open(file_path, 'r') as f:
            data = f.read().split()
    except FileNotFoundError:
        print(f"Không tìm thấy file {file_path}. Hãy đảm bảo bạn đã upload lên tab Files.")
        return

    ptr = 0
    num_instances = int(data[ptr]); ptr += 1

    for i in range(1, num_instances + 1):
        n = int(data[ptr]); ptr += 1
        m = int(data[ptr]); ptr += 1
        opt_val = float(data[ptr]); ptr += 1

        profits = [float(x) for x in data[ptr:ptr+n]]
        ptr += n

        weights = []
        for _ in range(m):
            weights.append([float(x) for x in data[ptr:ptr+n]])
            ptr += n

        capacities = [float(x) for x in data[ptr:ptr+m]]
        ptr += m

        if i == instance_to_solve:
            print(f"Đang giải bài toán thứ {i} trong file {file_path} ")
            print(f"Thông số: n={n}, m={m}")

            inst = MKPInstance(
                n=n, m=m,
                profits=np.array(profits),
                weights=np.array(weights),
                capacities=np.array(capacities)
            )

            result = hdp_solve(inst)

            selected_0_based = result["items"]
            selected_1_based = [j + 1 for j in selected_0_based]
            print(f"\nDANH SÁCH VẬT PHẨM ĐƯỢC CHỌN:")
            print(f" - Theo chỉ số Python (0..n-1): {selected_0_based}")
            print(f" - Theo thứ tự đề bài (1..n):   {selected_1_based}")

            x_vec = np.array(result["solution_vector"])
            print("\nSỬ DỤNG TÀI NGUYÊN:")
            for dim in range(inst.m):
                used = float(inst.weights[dim] @ x_vec)
                cap = float(inst.capacities[dim])
                print(f" - Chiều {dim + 1}: {used:.1f} / {cap:.1f} ({100 * used / cap:.1f}%)")

            if opt_val > 0:
                gap = abs(opt_val - result['profit']) / opt_val * 100
                print(f" - Độ lệch (Gap):               {gap:.4f}%")
            return

    print("Không tìm thấy bài toán yêu cầu.")


In [38]:
solve_from_uploaded_file("mknapcb1.txt", instance_to_solve=1)

Đang giải bài toán thứ 1 trong file mknapcb1.txt 
Thông số: n=100, m=5
    Giai đoạn 1: Nới lỏng Surrogate 
    Dual multipliers u = [0.1773 0.2392 0.2712 0.1922 0.12  ]
    Sức chứa surrogate C = 12656.6417
    Giai đoạn 2: DP Tối ưu (Beam Size: 10000)
    Tiến độ: 16/80 | Số trạng thái: 176 | Best: 12832.0
    Tiến độ: 32/80 | Số trạng thái: 619 | Best: 20398.0
    Tiến độ: 48/80 | Số trạng thái: 881 | Best: 21577.0
    Tiến độ: 64/80 | Số trạng thái: 1086 | Best: 21790.0
    Tiến độ: 80/80 | Số trạng thái: 886 | Best: 22896.0
    Giai đoạn 3: B&B Tối ưu (Best-First Search)
    B&B Hoàn tất: Lợi nhuận = 24304.00, Nodes = 194288

  KẾT QUẢ CUỐI CÙNG
  Lợi nhuận   : 24304.00
  Khả thi     : Có
  Thời gian   : 9.141s

DANH SÁCH VẬT PHẨM ĐƯỢC CHỌN:
 - Theo chỉ số Python (0..n-1): [1, 2, 6, 8, 10, 12, 17, 18, 23, 26, 28, 29, 31, 34, 43, 49, 56, 58, 61, 62, 65, 68, 70, 73, 76, 78, 84, 85, 92, 98]
 - Theo thứ tự đề bài (1..n):   [2, 3, 7, 9, 11, 13, 18, 19, 24, 27, 29, 30, 32, 35, 44, 50, 5

In [39]:
solve_from_uploaded_file("mknapcb1.txt", instance_to_solve=2)

Đang giải bài toán thứ 2 trong file mknapcb1.txt 
Thông số: n=100, m=5
    Giai đoạn 1: Nới lỏng Surrogate 
    Dual multipliers u = [0.2536 0.2056 0.1627 0.2231 0.155 ]
    Sức chứa surrogate C = 12813.6686
    Giai đoạn 2: DP Tối ưu (Beam Size: 10000)
    Tiến độ: 16/80 | Số trạng thái: 266 | Best: 11424.0
    Tiến độ: 32/80 | Số trạng thái: 552 | Best: 17073.0
    Tiến độ: 48/80 | Số trạng thái: 674 | Best: 20194.0
    Tiến độ: 64/80 | Số trạng thái: 837 | Best: 22221.0
    Tiến độ: 80/80 | Số trạng thái: 1024 | Best: 22954.0
    Giai đoạn 3: B&B Tối ưu (Best-First Search)
    B&B Hoàn tất: Lợi nhuận = 24225.00, Nodes = 246708

  KẾT QUẢ CUỐI CÙNG
  Lợi nhuận   : 24225.00
  Khả thi     : Có
  Thời gian   : 9.103s

DANH SÁCH VẬT PHẨM ĐƯỢC CHỌN:
 - Theo chỉ số Python (0..n-1): [3, 10, 20, 27, 28, 34, 36, 39, 41, 42, 45, 48, 49, 53, 56, 57, 61, 62, 64, 73, 74, 83, 88, 90, 91, 92, 93, 95, 99]
 - Theo thứ tự đề bài (1..n):   [4, 11, 21, 28, 29, 35, 37, 40, 42, 43, 46, 49, 50, 54, 57, 58,

In [41]:
solve_from_uploaded_file("mknapcb3.txt", instance_to_solve=1)

Đang giải bài toán thứ 1 trong file mknapcb3.txt 
Thông số: n=500, m=5
    Giai đoạn 1: Nới lỏng Surrogate 
    Dual multipliers u = [0.1976 0.2039 0.1925 0.2077 0.1983]
    Sức chứa surrogate C = 61327.6779
    Giai đoạn 2: DP Tối ưu (Beam Size: 10000)
    Tiến độ: 96/480 | Số trạng thái: 5109 | Best: 71003.0
    Tiến độ: 192/480 | Số trạng thái: 10000 | Best: 95238.0
    Tiến độ: 288/480 | Số trạng thái: 10000 | Best: 95238.0
    Tiến độ: 384/480 | Số trạng thái: 10000 | Best: 108824.0
    Tiến độ: 480/480 | Số trạng thái: 3918 | Best: 119009.0
    Giai đoạn 3: B&B Tối ưu (Best-First Search)
    B&B Hoàn tất: Lợi nhuận = 119963.00, Nodes = 936954

  KẾT QUẢ CUỐI CÙNG
  Lợi nhuận   : 119963.00
  Khả thi     : Có
  Thời gian   : 515.034s

DANH SÁCH VẬT PHẨM ĐƯỢC CHỌN:
 - Theo chỉ số Python (0..n-1): [0, 1, 4, 5, 9, 20, 23, 25, 29, 34, 38, 41, 43, 45, 56, 59, 61, 62, 70, 74, 75, 77, 78, 81, 83, 88, 93, 99, 108, 110, 115, 117, 121, 124, 131, 132, 134, 136, 139, 144, 147, 159, 165, 168, 1

In [42]:
solve_from_uploaded_file("mknapcb3.txt", instance_to_solve=2)

Đang giải bài toán thứ 2 trong file mknapcb3.txt 
Thông số: n=500, m=5
    Giai đoạn 1: Nới lỏng Surrogate 
    Dual multipliers u = [0.1936 0.2241 0.1714 0.2041 0.2068]
    Sức chứa surrogate C = 61868.3506
    Giai đoạn 2: DP Tối ưu (Beam Size: 10000)
    Tiến độ: 96/480 | Số trạng thái: 5873 | Best: 69320.0
    Tiến độ: 192/480 | Số trạng thái: 10000 | Best: 93347.0
    Tiến độ: 288/480 | Số trạng thái: 10000 | Best: 96441.0
    Tiến độ: 384/480 | Số trạng thái: 10000 | Best: 112987.0
    Tiến độ: 480/480 | Số trạng thái: 3104 | Best: 117114.0
    Giai đoạn 3: B&B Tối ưu (Best-First Search)
    B&B Hoàn tất: Lợi nhuận = 117685.00, Nodes = 113724

  KẾT QUẢ CUỐI CÙNG
  Lợi nhuận   : 117685.00
  Khả thi     : Có
  Thời gian   : 445.288s

DANH SÁCH VẬT PHẨM ĐƯỢC CHỌN:
 - Theo chỉ số Python (0..n-1): [0, 1, 4, 11, 16, 22, 23, 24, 26, 30, 32, 33, 34, 37, 41, 42, 44, 48, 51, 53, 57, 58, 59, 64, 65, 66, 71, 73, 74, 77, 91, 99, 104, 105, 109, 118, 122, 123, 124, 131, 143, 145, 146, 151, 156

In [43]:
solve_from_uploaded_file("mknapcb5.txt", instance_to_solve=1)

Đang giải bài toán thứ 1 trong file mknapcb5.txt 
Thông số: n=250, m=10
    Giai đoạn 1: Nới lỏng Surrogate 
    Dual multipliers u = [0.0382 0.1636 0.0837 0.1221 0.1015 0.1361 0.0866 0.0591 0.133  0.0761]
    Sức chứa surrogate C = 31277.9384
    Giai đoạn 2: DP Tối ưu (Beam Size: 10000)
    Tiến độ: 46/230 | Số trạng thái: 1332 | Best: 34224.0
    Tiến độ: 92/230 | Số trạng thái: 3326 | Best: 49994.0
    Tiến độ: 138/230 | Số trạng thái: 5022 | Best: 53276.0
    Tiến độ: 184/230 | Số trạng thái: 5500 | Best: 55593.0
    Tiến độ: 230/230 | Số trạng thái: 3015 | Best: 58222.0
    Giai đoạn 3: B&B Tối ưu (Best-First Search)
    B&B Hoàn tất: Lợi nhuận = 58896.00, Nodes = 1869404

  KẾT QUẢ CUỐI CÙNG
  Lợi nhuận   : 58896.00
  Khả thi     : Có
  Thời gian   : 157.408s

DANH SÁCH VẬT PHẨM ĐƯỢC CHỌN:
 - Theo chỉ số Python (0..n-1): [7, 11, 13, 15, 26, 27, 31, 33, 34, 36, 39, 45, 48, 49, 50, 51, 53, 55, 56, 57, 63, 65, 67, 69, 75, 76, 77, 80, 87, 93, 94, 103, 106, 112, 115, 116, 117, 126, 1

In [44]:
solve_from_uploaded_file("mknapcb5.txt", instance_to_solve=2)

Đang giải bài toán thứ 2 trong file mknapcb5.txt 
Thông số: n=250, m=10
    Giai đoạn 1: Nới lỏng Surrogate 
    Dual multipliers u = [0.0898 0.0866 0.1179 0.1125 0.0933 0.0906 0.1081 0.0988 0.123  0.0795]
    Sức chứa surrogate C = 30953.3324
    Giai đoạn 2: DP Tối ưu (Beam Size: 10000)
    Tiến độ: 46/230 | Số trạng thái: 1488 | Best: 34407.0
    Tiến độ: 92/230 | Số trạng thái: 3539 | Best: 49767.0
    Tiến độ: 138/230 | Số trạng thái: 5180 | Best: 54193.0
    Tiến độ: 184/230 | Số trạng thái: 5534 | Best: 56307.0
    Tiến độ: 230/230 | Số trạng thái: 2172 | Best: 57781.0
    Giai đoạn 3: B&B Tối ưu (Best-First Search)
    B&B Hoàn tất: Lợi nhuận = 58480.00, Nodes = 2961412

  KẾT QUẢ CUỐI CÙNG
  Lợi nhuận   : 58480.00
  Khả thi     : Có
  Thời gian   : 222.221s

DANH SÁCH VẬT PHẨM ĐƯỢC CHỌN:
 - Theo chỉ số Python (0..n-1): [1, 3, 4, 9, 10, 11, 12, 16, 17, 23, 27, 37, 38, 41, 42, 45, 49, 50, 51, 53, 57, 59, 62, 63, 64, 67, 68, 71, 86, 91, 93, 94, 96, 98, 100, 102, 103, 106, 109, 11

In [37]:
solve_from_uploaded_file("mknapcb7.txt", instance_to_solve=1)

Đang giải bài toán thứ 1 trong file mknapcb7.txt 
Thông số: n=100, m=30
    Giai đoạn 1: Nới lỏng Surrogate 
    Dual multipliers u = [5.52e-02 4.51e-02 1.00e-04 1.00e-04 5.95e-02 1.09e-01 7.36e-02 1.00e-04
 3.14e-02 5.30e-03 6.43e-02 8.63e-02 2.84e-02 1.00e-04 7.47e-02 3.80e-02
 3.58e-02 1.67e-02 4.35e-02 6.25e-02 1.00e-04 1.37e-02 2.83e-02 1.00e-04
 2.78e-02 2.94e-02 4.29e-02 1.63e-02 1.00e-04 1.19e-02]
    Sức chứa surrogate C = 12499.7468
    Giai đoạn 2: DP Tối ưu (Beam Size: 10000)
    Tiến độ: 16/80 | Số trạng thái: 165 | Best: 12269.0
    Tiến độ: 32/80 | Số trạng thái: 969 | Best: 17540.0
    Tiến độ: 48/80 | Số trạng thái: 3411 | Best: 18493.0
    Tiến độ: 64/80 | Số trạng thái: 6426 | Best: 19658.0
    Tiến độ: 80/80 | Số trạng thái: 7387 | Best: 20589.0
    Giai đoạn 3: B&B Tối ưu (Best-First Search)
    [Cảnh báo] Đã đạt giới hạn thời gian B&B.
    B&B Hoàn tất: Lợi nhuận = 21686.00, Nodes = 10885341

  KẾT QUẢ CUỐI CÙNG
  Lợi nhuận   : 21686.00
  Khả thi     : Có
  Thời g

In [45]:
solve_from_uploaded_file("mknapcb7.txt", instance_to_solve=2)

Đang giải bài toán thứ 2 trong file mknapcb7.txt 
Thông số: n=100, m=30
    Giai đoạn 1: Nới lỏng Surrogate 
    Dual multipliers u = [8.460e-02 7.130e-02 1.330e-02 1.000e-04 2.360e-02 3.040e-02 5.280e-02
 2.340e-02 1.000e-04 1.000e-04 3.090e-02 1.155e-01 1.000e-04 1.000e-04
 2.470e-02 4.320e-02 1.070e-02 1.000e-04 1.000e-04 7.270e-02 1.590e-02
 1.000e-04 6.560e-02 4.130e-02 1.000e-04 5.460e-02 1.000e-04 8.620e-02
 7.510e-02 6.350e-02]
    Sức chứa surrogate C = 12163.9169
    Giai đoạn 2: DP Tối ưu (Beam Size: 10000)
    Tiến độ: 16/80 | Số trạng thái: 298 | Best: 12796.0
    Tiến độ: 32/80 | Số trạng thái: 665 | Best: 17185.0
    Tiến độ: 48/80 | Số trạng thái: 1402 | Best: 18890.0
    Tiến độ: 64/80 | Số trạng thái: 7923 | Best: 19963.0
    Tiến độ: 80/80 | Số trạng thái: 8279 | Best: 20725.0
    Giai đoạn 3: B&B Tối ưu (Best-First Search)
    [Cảnh báo] Đã đạt giới hạn thời gian B&B.
    B&B Hoàn tất: Lợi nhuận = 21548.00, Nodes = 10672443

  KẾT QUẢ CUỐI CÙNG
  Lợi nhuận   : 21548

In [46]:
solve_from_uploaded_file("mknapcb9.txt", instance_to_solve=1)

Đang giải bài toán thứ 1 trong file mknapcb9.txt 
Thông số: n=500, m=30
    Giai đoạn 1: Nới lỏng Surrogate 
    Dual multipliers u = [0.0367 0.0127 0.0324 0.0265 0.0256 0.0309 0.0032 0.0517 0.0336 0.0489
 0.0443 0.0339 0.0438 0.0228 0.0377 0.0001 0.042  0.0123 0.0191 0.0209
 0.0726 0.0267 0.0236 0.0115 0.0409 0.0077 0.0549 0.0598 0.0724 0.0506]
    Sức chứa surrogate C = 62304.4330
    Giai đoạn 2: DP Tối ưu (Beam Size: 10000)
    Tiến độ: 96/480 | Số trạng thái: 5639 | Best: 71304.0
    Tiến độ: 192/480 | Số trạng thái: 10000 | Best: 90759.0
    Tiến độ: 288/480 | Số trạng thái: 10000 | Best: 90759.0
    Tiến độ: 384/480 | Số trạng thái: 10000 | Best: 93648.0
    Tiến độ: 480/480 | Số trạng thái: 9832 | Best: 114019.0
    Giai đoạn 3: B&B Tối ưu (Best-First Search)
    [Cảnh báo] Đã đạt giới hạn thời gian B&B.
    B&B Hoàn tất: Lợi nhuận = 114655.00, Nodes = 1429914

  KẾT QUẢ CUỐI CÙNG
  Lợi nhuận   : 114655.00
  Khả thi     : Có
  Thời gian   : 604.370s

DANH SÁCH VẬT PHẨM ĐƯỢC CHỌ

In [27]:
import pandas as pd
import numpy as np

data = [
    {"method": "HDP", "dataset": "100-5", "n": 100, "m": 5, "instance": 1, "profit": 24304.0, "time_sec": 10.830, "feasible": True, "num_items": 30, "avg_usage": 0.988, "min_usage": 0.975, "max_usage": 1.000},
    {"method": "HDP", "dataset": "100-5", "n": 100, "m": 5, "instance": 2, "profit": 24225.0, "time_sec": 9.013, "feasible": True, "num_items": 29, "avg_usage": 0.988, "min_usage": 0.978, "max_usage": 1.000},
    {"method": "HDP", "dataset": "500-5", "n": 500, "m": 5, "instance": 1, "profit": 119963.0, "time_sec": 497.763, "feasible": True, "num_items": 146, "avg_usage": 0.999, "min_usage": 0.999, "max_usage": 1.000},
    {"method": "HDP", "dataset": "500-5", "n": 500, "m": 5, "instance": 2, "profit": 117685.0, "time_sec": 429.302, "feasible": True, "num_items": 148, "avg_usage": 0.997, "min_usage": 0.995, "max_usage": 0.999},
    {"method": "HDP", "dataset": "250-10", "n": 250, "m": 10, "instance": 1, "profit": 58896.0, "time_sec": 155.325, "feasible": True, "num_items": 68, "avg_usage": 0.990, "min_usage": 0.967, "max_usage": 1.000},
    {"method": "HDP", "dataset": "250-10", "n": 250, "m": 10, "instance": 2, "profit": 58480.0, "time_sec": 218.095, "feasible": True, "num_items": 69, "avg_usage": 0.990, "min_usage": 0.972, "max_usage": 1.000},
    {"method": "HDP", "dataset": "100-30", "n": 100, "m": 30, "instance": 1, "profit": 21686.0, "time_sec": 615.527, "feasible": True, "num_items": 24, "avg_usage": 0.925, "min_usage": 0.763, "max_usage": 0.998},
    {"method": "HDP", "dataset": "100-30", "n": 100, "m": 30, "instance": 2, "profit": 21548.0, "time_sec": 620.731, "feasible": True, "num_items": 24, "avg_usage": 0.928, "min_usage": 0.783, "max_usage": 1.000},
    {"method": "HDP", "dataset": "500-30", "n": 500, "m": 30, "instance": 1, "profit": 114655.0, "time_sec": 603.493, "feasible": True, "num_items": 128, "avg_usage": 0.982, "min_usage": 0.951, "max_usage": 1.000}
]

df_final = pd.DataFrame(data)
pd.options.display.float_format = '{:.3f}'.format
display(df_final)

,method,dataset,n,m,instance,profit,time_sec,feasible,num_items,avg_usage,min_usage,max_usage
0,HDP,100-5,100,5,1,24304.000,10.830,True,30,0.988,0.975,1.000
1,HDP,100-5,100,5,2,24225.000,9.013,True,29,0.988,0.978,1.000
2,HDP,500-5,500,5,1,119963.000,497.763,True,146,0.999,0.999,1.000
3,HDP,500-5,500,5,2,117685.000,429.302,True,148,0.997,0.995,0.999
4,HDP,250-10,250,10,1,58896.000,155.325,True,68,0.990,0.967,1.000
5,HDP,250-10,250,10,2,58480.000,218.095,True,69,0.990,0.972,1.000
6,HDP,100-30,100,30,1,21686.000,615.527,True,24,0.925,0.763,0.998
7,HDP,100-30,100,30,2,21548.000,620.731,True,24,0.928,0.783,1.000
8,HDP,500-30,500,30,1,114655.000,603.493,True,128,0.982,0.951,1.000


In [28]:
df_final.to_csv('hdp.csv', index=False)
print("Đã xuất file hdp.csv thành công!")

Đã xuất file hdp.csv thành công!
